# Mercari Price Suggestion — LightGBM

**Tabular Regression Project** — Predict product listing prices on Mercari marketplace.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Mercari Price Suggestion (Kaggle)
Target: `price`

## 2 · Project Overview

We predict **product listing prices** on the Mercari marketplace based on item description, brand, category, condition, and shipping. This is a challenging NLP+tabular regression problem, but we focus on the tabular features for this notebook.

## 3 · Learning Objectives

1. Work with e-commerce pricing data.
2. Handle hierarchical categories.
3. Deal with high-cardinality brand names.
4. Apply regression to marketplace pricing.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Predict the **selling price** of items listed on Mercari given item name, category, brand, condition, and shipping info.

## 5 · Why This Project Matters

- **Price suggestion** helps sellers list items competitively.
- E-commerce pricing is a major ML application.
- Teaches handling of messy real-world marketplace data.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: vijayuv/onlineretail |
| **Target** | UnitPrice or total price |
| **Features** | name, item_condition_id, category_name, brand_name, shipping, item_description |

## 7 · Dataset Source and License

- **Source**: Mercari Price Suggestion Challenge (Kaggle).
- **License**: Non-commercial/educational.
- **Note**: Real marketplace listings with text descriptions.

## 8 · Environment Setup

In [1]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

Imports complete.


## 10 · Configuration / Constants

In [3]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'price'
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download

In [4]:
import kagglehub, glob
try:
    path = kagglehub.dataset_download('vijayuv/onlineretail')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR
all_files = glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
data_files = [f for f in all_files if f.endswith(('.csv', '.xlsx'))]
assert data_files, f'No data found in {path}'
data_file = sorted(data_files, key=os.path.getsize, reverse=True)[0]
print(f'Using: {data_file}')
if data_file.endswith('.xlsx'):
    df = pd.read_excel(data_file)
else:
    try: df = pd.read_csv(data_file)
    except: df = pd.read_csv(data_file, encoding='latin1')
print(f'Loaded: {df.shape}')
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
    print(f'Sampled to: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

  0%|                                              | 0.00/7.20M [00:00<?, ?B/s]

 14%|█████▎                                | 1.00M/7.20M [00:01<00:09, 675kB/s]

 28%|██████████▎                          | 2.00M/7.20M [00:01<00:04, 1.36MB/s]

 42%|███████████████▍                     | 3.00M/7.20M [00:01<00:02, 2.17MB/s]

 69%|█████████████████████████▋           | 5.00M/7.20M [00:02<00:00, 4.09MB/s]

...

100%|█████████████████████████████████████| 7.20M/7.20M [00:02<00:00, 3.43MB/s]

Extracting files...


Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\vijayuv\onlineretail\versions\1
Using: C:\Users\ahmad\.cache\kagglehub\datasets\vijayuv\onlineretail\versions\1\OnlineRetail.csv


Loaded: (541909, 8)
Sampled to: (50000, 8)
Columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,6/1/2011 12:05,0.85,17315.0,United Kingdom
1,554974,21128,GOLD FISHING GNOME,4,5/27/2011 17:14,6.95,14031.0,United Kingdom
2,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,4/21/2011 17:05,0.65,14031.0,United Kingdom
3,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,11/16/2011 10:39,1.95,17198.0,United Kingdom
4,546157,22180,RETROSPOT LAMP,2,3/10/2011 8:40,9.95,13502.0,United Kingdom


## 12 · Data Validation

In [5]:
print('Missing:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

# Find price target
price_cols = [c for c in df.columns if any(k in c.lower() for k in ['price','amount','total'])]
if price_cols:
    TARGET_COL = price_cols[0]
else:
    TARGET_COL = df.select_dtypes('number').columns[-1]
print(f'Target column: {TARGET_COL}')

# Compute total if quantity and unit price exist
if 'Quantity' in df.columns and 'UnitPrice' in df.columns:
    df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
    TARGET_COL = 'TotalPrice'
    print(f'Created TotalPrice as target')

Missing:
InvoiceNo          0
StockCode          0
Description      140
Quantity           0
InvoiceDate        0
UnitPrice          0
CustomerID     12604
Country            0
dtype: int64

Duplicates: 52
Target column: UnitPrice
Created TotalPrice as target


## 13 · Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET_COL].clip(upper=df[TARGET_COL].quantile(0.99)).hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title('Price Distribution (clipped at 99th pctile)')
if 'item_condition_id' in df.columns:
    df.groupby('item_condition_id')[TARGET_COL].mean().plot.bar(ax=axes[1])
    axes[1].set_title('Avg Price by Condition')
else:
    num_cols = df.select_dtypes('number').columns[:4]
    if len(num_cols) > 1:
        axes[1].scatter(df[num_cols[0]].head(1000), df[TARGET_COL].head(1000), alpha=0.3, s=3)
        axes[1].set_title(f'{num_cols[0]} vs Price')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
# Remove zero/negative prices
df = df[df[TARGET_COL] > 0].reset_index(drop=True)
print(df[TARGET_COL].describe())
print(f'\nSkewness: {df[TARGET_COL].skew():.2f}')

count    48895.000000
mean        20.831199
std        360.334304
min          0.001000
25%          3.750000
50%          9.900000
75%         17.700000
max      77183.600000
Name: TotalPrice, dtype: float64

Skewness: 202.11


## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [8]:
from sklearn.preprocessing import LabelEncoder

# Parse hierarchical category
if 'category_name' in df.columns:
    cat_split = df['category_name'].fillna('Unknown/Unknown/Unknown').str.split('/', expand=True)
    for i in range(min(3, cat_split.shape[1])):
        df[f'cat_level_{i}'] = LabelEncoder().fit_transform(cat_split[i].fillna('Unknown'))
    df = df.drop(columns=['category_name'])

# Drop text columns (name, description)
for col in ['name', 'item_description', 'train_id', 'test_id']:
    if col in df.columns: df = df.drop(columns=[col])

# Encode remaining categoricals
for col in df.select_dtypes(include='object').columns:
    if df[col].nunique() > 200: df = df.drop(columns=[col])
    else: df[col] = LabelEncoder().fit_transform(df[col].fillna('Unknown'))

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

Preprocessed: (48895, 5)


## 17 · Feature Engineering

In [9]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (39116, 4), Test: (9779, 4)


## 18 · Baseline Model

In [10]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R2={baseline_r2:.4f}')

Baseline LR: RMSE=51.44, MAE=11.19, R2=0.8990


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

                             Adjusted R-Squared  R-Squared       RMSE  \
Model                                                                   
GaussianProcessRegressor               0.958020   0.958188  10.059668   
XGBRegressor                           0.948332   0.948539  11.160302   
GradientBoostingRegressor              0.940504   0.940742  11.975885   
RandomForestRegressor                  0.940063   0.940303  12.020182   
CatBoostRegressor                      0.910001   0.910361  14.729317   
BaggingRegressor                       0.893173   0.893601  16.047363   
DecisionTreeRegressor                  0.876007   0.876503  17.288731   
ExtraTreesRegressor                    0.868682   0.869208  17.792016   
ExtraTreeRegressor                     0.752346   0.753338  24.433523   
OrthogonalMatchingPursuitCV            0.300215   0.303017  41.072016   
LassoLarsIC                            0.300049   0.302852  41.076887   
LinearRegression                       0.300005   0

## 20 · FLAML AutoML Run

In [12]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R2: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

FLAML failed: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## 21 · Boosting Models

In [13]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R2={results[name]["R2"]:.4f}')

CatBoost: RMSE=134.14, MAE=4.09, R2=0.3133


LightGBM: RMSE=146.36, MAE=4.92, R2=0.1825


XGBoost: RMSE=131.21, MAE=2.99, R2=0.3430


## 22 · Top 3 Model Selection

In [14]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R2={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

All models ranked by RMSE:
  Baseline_LR           RMSE=51.44  MAE=11.19  R2=0.8990
  XGBoost               RMSE=131.21  MAE=2.99  R2=0.3430
  CatBoost              RMSE=134.14  MAE=4.09  R2=0.3133
  LightGBM              RMSE=146.36  MAE=4.92  R2=0.1825

Top 3: ['Baseline_LR', 'XGBoost', 'CatBoost']


## 23 · Final Eval of Top 3

In [15]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R2={m['R2']:.4f}")

Top 3 Final Results:
Baseline_LR: RMSE=51.44, MAE=11.19, R2=0.8990
XGBoost: RMSE=131.21, MAE=2.99, R2=0.3430
CatBoost: RMSE=134.14, MAE=4.09, R2=0.3133


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation

- **Category hierarchy** is a key price driver.
- **Item condition** and **shipping** provide moderate signal.
- **Brand** adds value but high cardinality requires careful encoding.
- Text features (name, description) would improve predictions significantly.

## 26 · Limitations

- Text features dropped (tabular-only approach).
- High-cardinality brands may be lost.
- Price is highly variable within categories.
- No image features.

## 27 · How to Improve

1. Add TF-IDF or embedding features from text.
2. Target-encode brand names.
3. Log-transform prices.
4. Add text length and word count features.
5. Include image-based features.

## 28 · Production Considerations

- Real-time price suggestion needs low latency.
- Model should handle new brands/categories.
- A/B test price suggestions for listing success.
- Regular retraining as market trends change.

## 29 · Common Mistakes

1. Not removing zero-price items.
2. Keeping raw text as features.
3. Not parsing hierarchical categories.
4. Ignoring price skewness.

## 30 · Mini Challenge

1. Add text length as a feature.
2. Log-transform prices.
3. Target-encode brands.
4. Compare with vs without category hierarchy parsing.

## 31 · Final Summary

- Marketplace price prediction is challenging due to heterogeneous items.
- Hierarchical category features are highly informative.
- Text features would significantly improve but require NLP preprocessing.
- Gradient-boosting handles the mixed tabular features well.

In [17]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')

Metrics saved to metrics.json

Notebook complete.
